# Validation Test (large amplitude study) - Oscillating droplet

This notebook and the corresponding simulation setup notebook (OscillatingDroplet_DACH_Comparison_Run.ipynb) are part of published results (cf. V.B.) found in *From weakly to strongly nonlinear viscous drop shape oscillations: An analytical and numerical study* (https://doi.org/10.1103/PhysRevFluids.9.063601). 

In [ ]:
#r "BoSSSpad.dll"
//#r "../../../src/L4-application/BoSSSpad/bin/Release/net8.0/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

## Initialization tasks

Loading the `XNSE_Solver` and additional namespace:

In [2]:
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using NUnit.Framework;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Timestepping;

Overview on the available *Execution Queues* (aka. *Batch Processors*, aka. *Batch System*); these e.g. Linux HPC clusters on which compute jobs can be executed.

In [3]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


In [4]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

For this notebook (which is part of the BoSSS validation tests), a *default queue* is selected to run all jobs in the convergence study:

In [5]:
//var myBatch = ExecutionQueues[1];
var myBatch = (userName.Equals(@"FDY\jenkinsci")) ? GetDefaultQueue() : ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


Initialization of the Workflow management; there `OscillatingDropletPaper` is the project name which is used name all computations (aka. sessions):

In [6]:
wmg.Init("OscillatingDropletPaper", myBatch);

Project name is set to 'OscillatingDropletPaper'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\OscillatingDropletPaper'.


In [7]:
wmg.SetNameBasedSessionJobControlCorrelation();

In [ ]:
bool runShortTests = true; // set to 'false' to run all test cases for the whole period of time (up to 4.0), otherwise they will run only up to 0.5

## Sessions

In [9]:
static var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

## Verification of Initial Value data

Initial values and parameters for the simulation (intial droplet shape, Ohnesorg number, initial velocity)
were specified by project partner (TU Graz, Group Prof. Brenn).
Details can be found in the corresponding publication.

First, it is verified that the initial values chosen here actually match the specification.

In [10]:
//MultidimensionalArray[] ReferenceData = new MultidimensionalArray[5];
int[] modes = new int[] { 2, 3, 4 };
List<string> cases = new List<string>();
List<string> refDataProvided = new List<string>();
int nCase = 0;
// add simulations for mode 2
if(modes.Contains(2)) {
    cases.Add("m2_Oh01_eta03");
    refDataProvided.Add("m2_Oh01_eta03");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m2_Oh01_eta05");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m2_Oh01_eta06");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m2_Oh01_eta07");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
}
// add simulations for mode 3
if(modes.Contains(3)) {
    cases.Add("m3_Oh01_eta03");
    refDataProvided.Add("m3_Oh01_eta03");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m3_Oh01_eta05");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
}
// add simulations for mode 4
if(modes.Contains(4)) {
    cases.Add("m4_Oh01_eta02");
    refDataProvided.Add("m4_Oh01_eta02");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m4_Oh01_eta03");
    refDataProvided.Add("m4_Oh01_eta03");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m4_Oh01_eta05");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m4_Oh01_eta06");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m4_Oh01_eta07");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
}
int numCases = cases.Count;
numCases

case 1: m2_Oh01_eta03
case 2: m2_Oh01_eta05
case 3: m2_Oh01_eta06
case 4: m2_Oh01_eta07
case 5: m3_Oh01_eta03
case 6: m3_Oh01_eta05
case 7: m4_Oh01_eta02
case 8: m4_Oh01_eta03
case 9: m4_Oh01_eta05
case 10: m4_Oh01_eta06
case 11: m4_Oh01_eta07


11

Load reference data for all cases; These files contain two columns, i.e. the azimuth angle and the respective droplet radius.

In [11]:
MultidimensionalArray[] ReferenceData = new MultidimensionalArray[numCases];
for(int iCase = 0; iCase < numCases; iCase++) {
    if(refDataProvided.Contains(cases[iCase])) {
    if(userName.Equals(@"FDY\jenkinsci"))  
        ReferenceData[iCase] = IMatrixExtensions.LoadFromTextFile($"surfaceDrop_{cases[iCase]}.txt");
    else 
        ReferenceData[iCase] = IMatrixExtensions.LoadFromTextFile($"../data/InitialValues/{cases[iCase].Substring(0,2)}/surfaceDrop_{cases[iCase]}.txt");
    }
}

Analytical expressions for the reference data (see `setup.pdf`); this is to verify that the definition of Legendre
Functions resp. Polynomials in BoSSS actually matches the definition used by the TU Graz group.

In [12]:
Dictionary<string, Func<double, double>> casesRadius = new Dictionary<string, Func<double, double>>();

In [13]:
double caseR_m2eta07(double angle) {
   double radius = 0.895131 + 0.7*SphericalHarmonics.MyLegendre(2,0,Math.Cos(angle));
   return radius; 
}
casesRadius.Add("m2_Oh01_eta07", caseR_m2eta07);

In [14]:
double caseR_m2eta06(double angle) {
   double radius = 0.923706 + 0.6*SphericalHarmonics.MyLegendre(2,0,Math.Cos(angle));
   return radius; 
}
casesRadius.Add("m2_Oh01_eta06", caseR_m2eta06);

In [15]:
double caseR_m2eta05(double angle) {
   double radius = 0.947538 + 0.5*SphericalHarmonics.MyLegendre(2,0,Math.Cos(angle));
   return radius; 
}
casesRadius.Add("m2_Oh01_eta05", caseR_m2eta05);

In [16]:
double caseR_m2eta03(double angle) {
    double radius = 0.981486 + 0.3*SphericalHarmonics.MyLegendre(2,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m2_Oh01_eta03", caseR_m2eta03);

In [17]:
double caseR_m3eta05(double angle) {
    double radius = 0.964301 + 0.5*SphericalHarmonics.MyLegendre(3,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m3_Oh01_eta05", caseR_m3eta05);

In [18]:
double caseR_m3eta03(double angle) {
    double radius = 0.987144 + 0.3*SphericalHarmonics.MyLegendre(3,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m3_Oh01_eta03", caseR_m3eta03);

In [19]:
double caseR_m4eta07(double angle) {
    double radius = 0.943440 + 0.7*SphericalHarmonics.MyLegendre(4,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m4_Oh01_eta07", caseR_m4eta07);

In [20]:
double caseR_m4eta06(double angle) {
    double radius = 0.958674 + 0.6*SphericalHarmonics.MyLegendre(4,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m4_Oh01_eta06", caseR_m4eta06);

In [21]:
double caseR_m4eta05(double angle) {
    double radius = 0.971459 + 0.5*SphericalHarmonics.MyLegendre(4,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m4_Oh01_eta05", caseR_m4eta05);

In [22]:
double caseR_m4eta03(double angle) {
    double radius = 0.989838 + 0.3*SphericalHarmonics.MyLegendre(4,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m4_Oh01_eta03", caseR_m4eta03);

In [23]:
double caseR_m4eta02(double angle) {
    double radius = 0.995508 + 0.2*SphericalHarmonics.MyLegendre(4,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m4_Oh01_eta02", caseR_m4eta02);

Conversion to cartesian coordinates in order to match the data and verification against analytical expression:

In [24]:
double[][] refX = new double[numCases][];
double[][] refZ = new double[numCases][];
double[][] caseX = new double[numCases][];
double[][] caseZ = new double[numCases][];
for(int iCase = 0; iCase < numCases; iCase++) {
    if(refDataProvided.Contains(cases[iCase])) {
        double[] angle = ReferenceData[iCase].GetColumn(0);
        double[] radius = ReferenceData[iCase].GetColumn(1);
        
        double RadiusErrorNorm = 0.0;
        int I = angle.Length;
        double[] x1 = new double[I], z1 = new double[I];
        double[] cx1 = new double[I], cz1 = new double[I];
        for(int i = 0; i < I; i++) {
            x1[i] = Math.Sin(angle[i])*radius[i];
            z1[i] = Math.Cos(angle[i])*radius[i];
            
            double radius_expr = casesRadius[cases[iCase]](angle[i]);
            RadiusErrorNorm += (radius[i] - radius_expr).Pow2();     
            
            cx1[i] = Math.Sin(angle[i])*radius_expr;
            cz1[i] = Math.Cos(angle[i])*radius_expr;
        } 
        refX[iCase] = x1;
        refZ[iCase] = z1;
        caseX[iCase] = cx1;
        caseZ[iCase] = cz1;

        RadiusErrorNorm = RadiusErrorNorm.Sqrt();
        Console.WriteLine($"Comparison error for radius in case {iCase + 1}: {RadiusErrorNorm}");
        // Note: since the factors in `setup.pdf` are only provided up to 6 digits, an error threshold of 1e-5 seems reasonable.
        //Assert.LessOrEqual(RadiusErrorNorm, 1e-5, "Error in comparing reference data against Legendre polynomials in BoSSS");
    }
}

Comparison error for radius in case 1: 5.062870042266827E-06
Comparison error for radius in case 5: 0.0318269019380069
Comparison error for radius in case 7: 7.0258634464131154E-06
Comparison error for radius in case 8: 2.8677785870933128E-06


### Plot of Reference Data

In [25]:
int refCase = 0;
Plot(refX[refCase], refZ[refCase], "Ref-Case", ". black",
    caseX[refCase], caseZ[refCase], "Case", ". blue")

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -1.5 
 
 
 
 
 -1 
 
 
 
 
 -0.5 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 
 
 
 
 Ref-Case 
 
 
 Ref-Case 
 
 
 
	<path stroke='rgb( 0, 0, 0)' d='M716.2,34.7 L758.4,34.7 M64.2,56.6 L74.4,56.6 L84.7,56.7 L94.9,56.9 L105.2,57.1 L115.4,57.3
 L125.6,57.5 L135.8,57.8 L145.9,58.2 L156.0,58.5 L166.1,59.0 L176.2,59.4 L186.2,59.9 L196.2,60.4
 L206.1,61.0 L216.0,61.6 L225.9,62.3 L235.7,62.9 L245.4,63.6 L255.1,64.4 L264.7,65.2 L274.2,66.0
 L283.7,66.9 L293.1,67.8 L302.5,68.7 L311.7,69.7 L320.9,70.7 L330.0,71.7 L339.1,72.8 L348.0,73.9
 L356.9,75.0 L365.7,76.1 L374.3,77.3 L382.9,78.6 L391.4,79.8 L399.8,81.1 L408.1,82.4 L416.3,83.7
 L424.4,85.1 L432.4,86.5 L440.3,87.9 L448.1,89.3 L455.8,90.8 L463.4,92.3 L470.8,93.8 L478.2,95.4
 L485.4,96.9 L492.5,98.5 L499.5,100.1 L506.4,101.7 L513.2,103.4 L519.8,105.1 L526.4,106.7 L532.8,108.4
 L539.1,110.2 L545.3,111.9 L551.3,113.7 L557.2,115.4 L563.1,117.2 L568.7,119.0 L574.3,120.8 L579.8,122.6
 L585.1,124.5 L590.3,126.3 L595.4,128.2 L600.3,130.0 L605.2,131.9 L609.9,133.8 L614.5,135.7 L619.0,137.6
 L623.3,139.5 L627.6,141.4 L631.7,143.3 L635.7,145.3 L639.6,147.2 L643.4,149.1 L647.0,151.1 L650.6,153.0
 L654.0,155.0 L657.4,156.9 L660.6,158.8 L663.7,160.8 L666.7,162.7 L669.6,164.7 L672.4,166.6 L675.1,168.6
 L677.7,170.5 L680.2,172.5 L682.6,174.4 L684.9,176.3 L687.1,178.3 L689.2,180.2 L691.2,182.1 L693.2,184.0
 L695.0,186.0 L696.8,187.9 L698.5,189.8 L700.1,191.7 L701.6,193.6 L703.1,195.4 L704.4,197.3 L705.7,199.2
 L707.0,201.1 L708.1,202.9 L709.2,204.8 L710.2,206.6 L711.2,208.4 L712.1,210.3 L713.0,212.1 L713.7,213.9
 L714.5,215.7 L715.2,217.5 L715.8,219.3 L716.4,221.0 L716.9,222.8 L717.4,224.5 L717.8,226.3 L718.2,228.0
 L718.6,229.8 L719.0,231.5 L719.3,233.2 L719.5,234.9 L719.8,236.6 L720.0,238.3 L720.1,239.9 L720.3,241.6
 L720.4,243.3 L720.5,244.9 L720.6,246.6 L720.7,248.2 L720.7,249.8 L720.8,251.4 L720.8,253.1 L720.8,254.7
 L720.8,256.3 L720.8,257.9 L720.8,259.4 L720.7,261.0 L720.7,262.6 L720.7,264.2 L720.6,265.7 L720.6,267.3
 L720.5,268.8 L720.5,270.4 L720.4,271.9 L720.4,273.5 L720.4,275.0 L720.3,276.5 L720.3,278.1 L720.2,279.6
 L720.2,281.1 L720.2,282.6 L720.1,284.2 L720.1,285.7 L720.1,287.2 L720.1,288.7 L720.1,290.2 L720.1,291.7
 L720.1,293.3 L720.1,294.8 L720.1,296.3 L720.2,297.8 L720.2,299.3 L720.2,300.9 L720.3,302.4 L720.3,303.9
 L720.3,305.5 L720.4,307.0 L720.4,308.5 L720.5,310.1 L720.5,311.6 L720.6,313.2 L720.6,314.7 L720.7,316.3
 L720.7,317.9 L720.7,319.4 L720.8,321.0 L720.8,322.6 L720.8,324.2 L720.8,325.8 L720.8,327.4 L720.8,329.0
 L720.7,330.6 L720.7,332.2 L720.6,333.9 L720.5,335.5 L720.4,337.2 L720.3,338.8 L720.2,340.5 L720.0,342.2
 L719.8,343.8 L719.6,345.5 L719.3,347.2 L719.0,348.9 L718.7,350.7 L718.3,352.4 L717.9,354.1 L717.5,355.9
 L717.0,357.6 L716.5,359.4 L715.9,361.2 L715.3,362.9 L714.6,364.7 L713.9,366.5 L713.1,368.3 L712.2,370.2
 L711.4,372.0 L710.4,373.8 L709.4,375.6 L708.3,377.5 L707.1,379.3 L705.9,381.2 L704.6,383.1 L703.3,385.0
 L701.8,386.8 L700.3,388.7 L698.7,390.6 L697.1,392.5 L695.3,394.4 L693.5,396.4 L691.6,398.3 L689.5,400.2
 L687.4,402.1 L685.2,404.1 L683.0,406.0 L680.6,407.9 L678.1,409.9 L675.5,411.8 L672.8,413.8 L670.0,415.7
 L667.2,417.6 L664.2,419.6 L661.1,421.5 L657.9,423.5 L654.6,425.4 L651.1,427.4 L647.6,429.3 L644.0,431.3
 L640.2,433.2 L636.3,435.1 L632.3,437.0 L628.2,439.0 L624.0,440.9 L619.7,442.8 L615.2,444.7 L610.6,446.6
 L605.9,448.5 L601.1,450.4 L596.2,452.2 L591.1,454.1 L585.9,455.9 L580.6,457.8 L575.2,459.6 L569.6,461.4
 L564.0,463.2 L558.2,465.0 L552.3,466.8 L546.2,468.5 L540.1,470.3 L533.8,472.0 L527.4,473.7 L520.9,475.4
 L514.3,47

### Matching of the Spherical Harmonics against the provided Data

In [26]:
Dictionary<string, Formula> casesPhi = new Dictionary<string, Formula>();

In [27]:
var Phi_m2eta07_Init = new Formula(
"Phi1",
false,
"using ilPSP.Utils; " + 
"double Phi1(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.895131*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.7*SphericalHarmonics.MyRealSpherical(2, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"}");
casesPhi.Add("m2_Oh01_eta07", Phi_m2eta07_Init);

In [28]:
var Phi_m2eta06_Init = new Formula(
"Phi4",
false,
"using ilPSP.Utils; " + 
"double Phi4(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.923706*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.6*SphericalHarmonics.MyRealSpherical(2, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m2_Oh01_eta06", Phi_m2eta06_Init);

In [29]:
var Phi_m2eta05_Init = new Formula(
"Phi4",
false,
"using ilPSP.Utils; " + 
"double Phi4(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.947538*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.5*SphericalHarmonics.MyRealSpherical(2, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m2_Oh01_eta05", Phi_m2eta05_Init);

In [30]:
var Phi_m2eta03_Init = new Formula(
"Phi4",
false,
"using ilPSP.Utils; " + 
"double Phi4(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.981486*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.3*SphericalHarmonics.MyRealSpherical(2, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m2_Oh01_eta03", Phi_m2eta03_Init);

In [31]:
var Phi_m3eta07_Init = new Formula(
"Phi2",
false,
"using ilPSP.Utils; " + 
"double Phi2(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.930122*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.7*SphericalHarmonics.MyRealSpherical(3, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m3_Oh01_eta07", Phi_m3eta07_Init);

In [32]:
var Phi_m3eta06_Init = new Formula(
"Phi2",
false,
"using ilPSP.Utils; " + 
"double Phi2(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.948619*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.6*SphericalHarmonics.MyRealSpherical(3, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m3_Oh01_eta06", Phi_m3eta06_Init);

In [33]:
var Phi_m3eta05_Init = new Formula(
"Phi2",
false,
"using ilPSP.Utils; " + 
"double Phi2(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.964301*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.5*SphericalHarmonics.MyRealSpherical(3, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m3_Oh01_eta05", Phi_m3eta05_Init);

In [34]:
var Phi_m3eta03_Init = new Formula(
"Phi2",
false,
"using ilPSP.Utils; " + 
"double Phi2(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.987144*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.3*SphericalHarmonics.MyRealSpherical(3, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m3_Oh01_eta03", Phi_m3eta03_Init);

In [35]:
var Phi_m4eta07_Init = new Formula(
"Phi3",
false,
"using ilPSP.Utils; " + 
"double Phi3(double[] X) { " +    
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.943440*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.7*SphericalHarmonics.MyRealSpherical(4, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m4_Oh01_eta07", Phi_m4eta07_Init);

In [36]:
var Phi_m4eta06_Init = new Formula(
"Phi3",
false,
"using ilPSP.Utils; " + 
"double Phi3(double[] X) { " +    
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.958674*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.6*SphericalHarmonics.MyRealSpherical(4, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m4_Oh01_eta06", Phi_m4eta06_Init);

In [37]:
var Phi_m4eta05_Init = new Formula(
"Phi3",
false,
"using ilPSP.Utils; " + 
"double Phi3(double[] X) { " +    
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.971459*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.5*SphericalHarmonics.MyRealSpherical(4, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m4_Oh01_eta05", Phi_m4eta05_Init);

In [38]:
var Phi_m4eta03_Init = new Formula(
"Phi3",
false,
"using ilPSP.Utils; " + 
"double Phi3(double[] X) { " +    
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.989838*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.3*SphericalHarmonics.MyRealSpherical(4, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m4_Oh01_eta03", Phi_m4eta03_Init);

In [39]:
var Phi_m4eta02_Init = new Formula(
"Phi3",
false,
"using ilPSP.Utils; " + 
"double Phi3(double[] X) { " +    
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.995508*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.2*SphericalHarmonics.MyRealSpherical(4, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m4_Oh01_eta02", Phi_m4eta02_Init);

In [40]:
for(int iCase = 0; iCase < numCases; iCase++) {
    if(refDataProvided.Contains(cases[iCase])) {
        double[] angle = ReferenceData[iCase].GetColumn(0);
        int I = angle.Length;
        
        double PhiErr = 0;
        for(int i = 0; i < I; i++) {
            double radius_expr = casesRadius[cases[iCase]](angle[i]);    
            double x1 = Math.Sin(angle[i])*radius_expr;
            double z1 = Math.Cos(angle[i])*radius_expr;
        
            PhiErr += casesPhi[cases[iCase]].Evaluate(new Vector(x1, 0, z1), 0.0).Abs();
        }
        Console.WriteLine($"Phi error for case {iCase}: {PhiErr}");
        Assert.LessOrEqual(PhiErr, 1e-10, "Level-Set function is not zero at desired surface.");
        
        Assert.IsTrue(casesPhi[cases[iCase]].Evaluate(new Vector(1e-5, 1e-5, 1e-5), 0.0) < 0, "Inside must be phase A/negative");
        Assert.IsTrue(casesPhi[cases[iCase]].Evaluate(new Vector(1e+1, 1e+1, 1e+1), 0.0) > 0, "Outside must be phase B/positive");
    }
}

Phi error for case 0: 1.7430501486614958E-14
Phi error for case 4: 2.0095036745715333E-14
Phi error for case 6: 1.965094753586527E-14
Phi error for case 7: 2.6978419498391304E-14


## Grid Creation

### Quater-Domain grid
(Symmetry planes at $x = 0$ and $y = 0$)

In [41]:
int[] Resolutions = new int[] { 7 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];
double scale = 1.0;
for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"OscillatingDroplet3D_{Res}x{Res}x{2*Res}_wallBC_quarterDomain";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    //cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = GenericBlas.Linspace(0, 3*scale, Res + 1);
        double[] yNodes = xNodes;
        double[] zNodes = GenericBlas.Linspace(-3*scale, 3*scale, Res*2 + 1);
        
        var grd = Grid3D.Cartesian3DGrid(xNodes, yNodes, zNodes);
        grd.Name = GridName;
        
        grd.DefineEdgeTags(delegate(Vector X) {
            string ret = null;
            if(X.x.Abs() <= 1e-8 || X.y.Abs() <= 1.0e-8)
                ret = IncompressibleBcType.SlipSymmetry.ToString();
            else
                ret = IncompressibleBcType.Wall.ToString();
            return ret;
        });        
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Opening existing database 'C:\BoSSS-localJobs\OscillatingDropletPaper'.
Grid already found in database - identifid by name OscillatingDroplet3D_7x7x14_wallBC_quarterDomain


## Setup of control objects for all solver runs

In [48]:
public class StudySettings {

    public double Oh;

    public int m;
    public double eta0;

    public string caseName;

    public int maxL;

    public double t_end;
    public double dt;
    public int savePeriod;
    public int logPeriod;

    public StudySettings(int mode, double deformParam, int maxLmode, int savePer = 10, int logPer = 2) {

        Oh = 0.1;
        m = mode;
        eta0 = deformParam;

        caseName = $"m{m}_Oh{Oh.ToString("0.##").Replace(".", "")}_eta{eta0.ToString("0.###").Replace(".", "")}";

        maxL = maxLmode;

        t_end = 4.0;
        dt = 0.005;

        savePeriod = savePer;   
        logPeriod = logPer;

    }

}

In [50]:
List<StudySettings> allStudies = new List<StudySettings>();

allStudies.Add(new StudySettings(2, 0.3, 12));    // case 1.4
allStudies.Add(new StudySettings(2, 0.5, 20));    // case 1.5
allStudies.Add(new StudySettings(2, 0.6, 20));    // case 1.6
allStudies.Add(new StudySettings(2, 0.7, 20));    // case 1.7
allStudies.Add(new StudySettings(3, 0.3, 12));    // case 2.3
allStudies.Add(new StudySettings(3, 0.5, 24));    // case 2.4
allStudies.Add(new StudySettings(4, 0.2, 20));    // case 3.3
allStudies.Add(new StudySettings(4, 0.3, 20));    // case 3.4
allStudies.Add(new StudySettings(4, 0.5, 24));    // case 3.5
allStudies.Add(new StudySettings(4, 0.6, 24));    // case 3.6
allStudies.Add(new StudySettings(4, 0.7, 30));    // case 3.7

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

foreach(var study in allStudies) {

    var C = new XNSE_Control();

    int k = 3;
    C.SetDGdegree(k);


    // physical parameters
    // ===================
    C.PhysicalParameters.rho_A = 1;
    C.PhysicalParameters.rho_B = 0.001;
    C.PhysicalParameters.mu_A = study.Oh;
    C.PhysicalParameters.mu_B = study.Oh/1000.0;
    C.PhysicalParameters.Sigma = 1;

    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = true;
    

    // set grid
    // ========
    C.SetGrid(Grids[0]);

    
    // initial conditions
    // ==================
    C.InitialValues.Add("Phi", casesPhi[study.caseName]);


    // misc. solver options
    // ====================
    C.LinearSolver = LinearSolverCode.direct_mumps.GetConfig(); 
    C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
    C.NonLinearSolver.ConvergenceCriterion = 1e-9;  
    C.NonLinearSolver.MaxSolverIterations = 50;
    C.NonLinearSolver.MinSolverIterations = 3;


    // level-set
    // =========
    C.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

    C.CutCellQuadratureType = CutCellQuadratureMethod.Saye;
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

    C.AdaptiveMeshRefinement = true;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = 1 });


    // Timestepping
    // ============    
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.dtFixed = study.dt;
    C.Endtime = runShortTests ? 0.5 : study.t_end;
    C.NoOfTimesteps = (int)(study.t_end / study.dt);
    

    
    C.PostprocessingModules.Add(new SphericalHarmonicsLogging() { SolverStage = 2, MaxL = study.maxL, RotSymmetric = true, LogPeriod = study.logPeriod });
    C.PostprocessingModules.Add(new DropletMetricsLogging() { SolverStage = 2, AxisSymmetric = true, LogPeriod = study.logPeriod });
    C.PostprocessingModules.Add(new EnergyLogging() { SolverStage = 2, LogPeriod = study.logPeriod });

    C.saveperiod =  study.savePeriod;
    
    //C.TracingNamespaces = "*";

    C.SessionName = $"OD3D_{study.caseName}_stagnantInit";

    //Console.WriteLine($"Setting up case: {C.SessionName}");
    
    Controls.Add(C);
    
}

In [53]:
foreach (var ctrl in Controls) {
    //Console.WriteLine($"{ctrl.SessionName}");
    Console.WriteLine($"{ctrl.SessionName}: tend = {ctrl.Endtime}");
}

OD3D_m2_Oh01_eta03_stagnantInit: tend = 1
OD3D_m2_Oh01_eta05_stagnantInit: tend = 1
OD3D_m2_Oh01_eta06_stagnantInit: tend = 1
OD3D_m2_Oh01_eta07_stagnantInit: tend = 1
OD3D_m3_Oh01_eta03_stagnantInit: tend = 1
OD3D_m3_Oh01_eta05_stagnantInit: tend = 1
OD3D_m4_Oh01_eta02_stagnantInit: tend = 1
OD3D_m4_Oh01_eta03_stagnantInit: tend = 1
OD3D_m4_Oh01_eta05_stagnantInit: tend = 1
OD3D_m4_Oh01_eta06_stagnantInit: tend = 1
OD3D_m4_Oh01_eta07_stagnantInit: tend = 1


In [46]:
int NoCtrls = Controls.Count();
NoCtrls

11

## Launch Jobs

In [ ]:
List<Job> jobs = new List<Job>();

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.RetryCount = 1;
    oneJob.Activate(myBatch);

    jobs.Add(oneJob);
}

Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh8	3/23/2020 9:35:37 PM	bdd58f86..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh16	3/24/2020 2:29:40 PM	eed03bcf..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-W

## Wait for Completion and Check Job Status

In [ ]:
int NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress 
                                    || job.Status == JobStatus.PendingInExecutionQueue
                                    || job.Status == JobStatus.PreActivation).Count();
NoInProgress

0

In [ ]:
double maxWait = 1 * 24;    // in hours
int waitingCount = 0;
double waitingInterval = 6; // in hours
while (NoInProgress > 0 && waitingCount * waitingInterval < maxWait) {
    wmg.BlockUntilAllJobsTerminate(3600*waitingInterval); // wait one day for the jobs to finish
    NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress).Count();
    waitingCount++;
}

In [ ]:
var FailedSess = jobs.Where(job => job.Status == JobStatus.FailedOrCanceled);
FailedSess

0

In [ ]:
int NoFailed = FailedSess.Count();
NoFailed

In [ ]:
var SuccessSess = jobs.Where(job => job.Status == JobStatus.FinishedSuccessful);
SuccessSess

In [ ]:
int NoSuccess = SuccessSess.Count();
NoSuccess

17

In [ ]:
// check for restart sessions
if (NoFailed > 0) {
    foreach (var job in jobs) {
        //var job = ctrl.GetJob();
        if (job.Status == JobStatus.FailedOrCanceled) {
            var studySess = sessions.Where(sess => sess.Name.Contains(job.Name));
            int studyCount = studySess.Count();

            if (studyCount > 1) {
                Console.WriteLine("Restart session found! Take last run");       
                bool success = studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single().SuccessfulTermination;
                if (success) {
                    Console.WriteLine($"Restart session {job.Name}_restart{studyCount-1} was successful.");
                    NoFailed--;
                    NoSuccess++;
                } else {
                    Console.WriteLine($"Restart session {job.Name}_restart{studyCount-1} also failed.");
                }
            } else if (studyCount == 1) {
                Console.WriteLine($"No restart session found. {job.Name} might to be restarted.");
            } else { // studyCount == 0
                throw new ApplicationException($"No session found for {job.Name}!"); // should not happen
            } 
        }
    }

}

In [ ]:
//NUnit.Framework.Assert.AreEqual(0, NoFailed, $"Some simulations failed.");

In [ ]:
//NUnit.Framework.Assert.AreEqual(NoCtrls, NoSuccess, $"Not all simulations finished successfully.");